# Ballistics pilot: do measurable properties of how a model sees predict which partner rescues its missed severe disease?

Follows `04_results/ballistics_pilot_plan.json` (written before anything here was trained or measured).

**Stages** (each resumable; rerun any cell after a disconnect):
1. Data: EyePACS train/val images, confirmation-set images, 2,000-image fingerprint subset (and its degraded versions).
2. Zoo: 22 backbones, one short identical recipe per family (CNN 5 epochs AdamW 1e-4; transformer/mixer 5 epochs AdamW 3e-5 with warm-up, clip 1.0), one seed. Each writes a checkpoint and confirmation-set predictions.
3. Calibration gate: pairwise severe co-miss of the cheap anchors vs their fully trained counterparts (Spearman must be at least 0.6, else stop).
4. Fingerprints (no labels): frequency, channel, degradation response vectors; occlusion maps; CKA (trained and pretrained); pretrained feature-shift responses.
5. Analysis: pair-level prediction of severe co-miss, leave-one-family-out, added value over accuracy and spec sheet, and the clinical partner-choice test.

A100, High-RAM, `HF_TOKEN`. Roughly 2.5 to 3.5 hours end to end.

In [1]:
# Cell 1: setup
from google.colab import auth, drive, userdata
auth.authenticate_user(); drive.mount('/content/drive')
!pip -q install timm transformers
import io, os, glob, json, time, zipfile, math, random, datetime, itertools, hashlib
import numpy as np, pandas as pd, torch, torch.nn as nn, timm
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))
svc = gbuild('drive', 'v3')
BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'
SPLITS, PREDS, RESULTS = f'{BASE}/01_splits', f'{BASE}/03_predictions', f'{BASE}/04_results'
ZOO_CK, ZOO_OUT = f'{BASE}/02_checkpoints/zoo', f'{BASE}/04_results/ballistics'
for d in (ZOO_CK, ZOO_OUT, f'{ZOO_OUT}/fingerprints', f'{ZOO_OUT}/preds'): os.makedirs(d, exist_ok=True)
SPLIT_IDS = {'train': '1FU30c6vRYMLqX8h9z2F58iEqow1Zx8NQ', 'val': '1Z9tFlAGe0j4RVRgG9-6LnhTrukvFy87y'}
IMAGE_ZIPS = {'batch1': '11WspNNHS73OadbW-sqy7BSzcrsBkX1d6', 'batch2': '1zG7rHjm8Ty9LHG4LqMzgnOzrT_90VO9H', 'batch3': '1L88vUNSxWjdGPBquhknZhafcnXXRd3ES',
              'batch4': '1-FrpGB74qEGzDvXlA9FhmB2OeeZkKPAN', 'batch5': '1mEIrSGVExZ6zmxnRLxyFab5yB1kH5YBT'}
RETFOUND_REPO = 'iszt/RETFound_mae_meh'; NW = min(8, os.cpu_count()); dev = 'cuda'
def dl(fid, path):
    with io.FileIO(path, 'wb') as fh:
        d = MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid), chunksize=50 * 1024 * 1024); done = False
        while not done: _, done = d.next_chunk()
torch.backends.cudnn.benchmark = True; torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
print(torch.cuda.get_device_name(0))

Mounted at /content/drive
NVIDIA A100-SXM4-40GB


In [2]:
# Cell 2: the zoo (name, timm id, family, pretraining data, objective, recipe). Anchors match the fully trained paper models.
ZOO = [
 ('resnet50',        'resnet50',                                  'resnet',      'in1k',    'supervised', 'cnn'),   # anchor
 ('resnet50_swsl',   'resnet50.fb_swsl_ig1b_ft_in1k',             'resnet',      'ig1b',    'semi_weak',  'cnn'),   # anchor
 ('resnext50',       'resnext50_32x4d.a1h_in1k',                  'resnet',      'in1k',    'supervised', 'cnn'),
 ('regnety_032',     'regnety_032.ra_in1k',                       'regnet',      'in1k',    'supervised', 'cnn'),
 ('densenet121',     'densenet121',                               'densenet',    'in1k',    'supervised', 'cnn'),   # anchor
 ('efficientnet_b3', 'efficientnet_b3',                           'efficientnet','in1k',    'supervised', 'cnn'),   # anchor
 ('effnetv2_s',      'tf_efficientnetv2_s.in21k_ft_in1k',         'efficientnet','in21k',   'supervised', 'cnn'),
 ('mobilenetv3',     'mobilenetv3_large_100.ra_in1k',             'mobilenet',   'in1k',    'supervised', 'cnn'),
 ('convnext_t',      'convnext_tiny.fb_in1k',                     'convnext',    'in1k',    'supervised', 'cnn'),
 ('convnext_t22k',   'convnext_tiny.fb_in22k_ft_in1k',            'convnext',    'in22k',   'supervised', 'cnn'),
 ('inception_v3',    'inception_v3.tv_in1k',                      'inception',   'in1k',    'supervised', 'cnn'),
 ('vgg16_bn',        'vgg16_bn.tv_in1k',                          'vgg',         'in1k',    'supervised', 'cnn'),
 ('vit_b',           'vit_base_patch16_224',                      'vit',         'in21k',   'supervised', 'tfm'),   # anchor
 ('vit_b_mae',       'vit_base_patch16_224.mae',                  'vit',         'in1k',    'mae',        'tfm'),   # anchor
 ('vit_b_dinov2',    'vit_base_patch14_dinov2.lvd142m',           'vit',         'lvd142m', 'dino',       'tfm'),   # anchor
 ('vit_b_clip',      'vit_base_patch16_clip_224.openai_ft_in1k',  'vit',         'wit400m', 'contrastive','tfm'),
 ('vit_s',           'vit_small_patch16_224.augreg_in21k_ft_in1k','vit',         'in21k',   'supervised', 'tfm'),
 ('deit_s',          'deit_small_patch16_224.fb_in1k',            'deit',        'in1k',    'supervised', 'tfm'),
 ('swin_t',          'swin_tiny_patch4_window7_224.ms_in1k',      'swin',        'in1k',    'supervised', 'tfm'),
 ('beitv2_b',        'beitv2_base_patch16_224.in1k_ft_in1k',      'beit',        'in1k',    'mim',        'tfm'),
 ('mixer_b16',       'mixer_b16_224.goog_in21k_ft_in1k',          'mixer',       'in21k',   'supervised', 'tfm'),
 ('retfound',        'retfound',                                  'retfound',    'retina',  'mae',        'tfm'),   # anchor
]
Z = pd.DataFrame(ZOO, columns=['name','timm_id','family','pretrain_data','objective','recipe']).set_index('name')
ANCHOR_FULL = {'resnet50': 'resnet50_5class', 'resnet50_swsl': 'resnet50_swsl_5class', 'densenet121': 'densenet121_5class', 'efficientnet_b3': 'efficientnet_b3_5class',
               'vit_b': 'vit_base_5class', 'vit_b_mae': 'vit_base_mae_5class', 'vit_b_dinov2': 'vit_base_dinov2_5class', 'retfound': 'retfound_5class'}
RECIPE = {'cnn': {'epochs': 5, 'lr': 1e-4, 'wd': 1e-4, 'warmup': 0, 'clip': None, 'bs': 32},
          'tfm': {'epochs': 5, 'lr': 3e-5, 'wd': 0.05, 'warmup': 1, 'clip': 1.0, 'bs': 32}}
SEED = 20261401
from transformers import AutoModel, AutoConfig
class RF(nn.Module):
    def __init__(self, nc=5, pretrained=True):
        super().__init__(); cfg = AutoConfig.from_pretrained(RETFOUND_REPO)
        self.base = AutoModel.from_pretrained(RETFOUND_REPO, add_pooling_layer=False) if pretrained else AutoModel.from_config(cfg, add_pooling_layer=False)
        self.fc_norm = nn.LayerNorm(cfg.hidden_size, eps=1e-6); self.classifier = nn.Linear(cfg.hidden_size, nc)
        nn.init.trunc_normal_(self.classifier.weight, std=0.01); nn.init.zeros_(self.classifier.bias)
    def features(self, x): return self.fc_norm(self.base(pixel_values=x).last_hidden_state[:, 1:].mean(1))
    def forward(self, x): return self.classifier(self.features(x))
def build(name, pretrained=True, nc=5):
    tid = Z.loc[name, 'timm_id']
    if tid == 'retfound': return RF(nc, pretrained)
    kw = {'img_size': 224} if Z.loc[name, 'recipe'] == 'tfm' and not tid.startswith(('swin', 'mixer')) else {}
    return timm.create_model(tid, pretrained=pretrained, num_classes=nc, **kw)
def penultimate(m, x):
    if isinstance(m, RF): return m.features(x)
    return m.forward_head(m.forward_features(x), pre_logits=True)
for name in Z.index:   # preflight: every model builds and runs at 224
    m = build(name).eval()
    with torch.no_grad(): o = m(torch.zeros(1, 3, 224, 224)); f = penultimate(m, torch.zeros(1, 3, 224, 224))
    assert o.shape == (1, 5), name; print(f'{name:15s} ok  feat {tuple(f.shape)}'); del m
print(len(Z), 'models')

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

resnet50        ok  feat (1, 2048)


model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

...

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

retfound        ok  feat (1, 1024)
22 models


In [3]:
# Cell 3: images. EyePACS train/val (API zips), confirmation set (Drive zips), fingerprint subset
from PIL import Image, ImageFilter
from torchvision import transforms
os.makedirs('/content/img', exist_ok=True); splits = {}
for s, fid in SPLIT_IDS.items():
    if not os.path.exists(f'split_{s}.csv'): dl(fid, f'split_{s}.csv')
    splits[s] = pd.read_csv(f'split_{s}.csv')
if not glob.glob('/content/img/**/*.jp*g', recursive=True):
    for nm, fid in IMAGE_ZIPS.items():
        z = f'/content/{nm}.zip'; dl(fid, z)
        with zipfile.ZipFile(z) as zf: zf.extractall('/content/img')
        os.remove(z)
INDEX = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob('/content/img/**/*.*', recursive=True) if p.lower().endswith(('.jpg','.jpeg','.png'))}
fr = pd.read_csv(f'{SPLITS}/eyepacs_split_fresh_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
FR = pd.DataFrame({'image_id': fr.image_id, 'level': fr[next(c for c in ('level','grade') if c in fr.columns)].astype(int), 'patient_id': fr.patient_id.astype(str)})
FR['binary_label'] = (FR.level >= 2).astype(int)
os.makedirs('/content/fresh', exist_ok=True)
if len(glob.glob('/content/fresh/**/*.*', recursive=True)) < len(FR):
    for z in sorted(glob.glob(f'{BASE}/06_fresh_images/eyepacs_fresh_batch*_v2.zip')):
        with zipfile.ZipFile(z) as zf: zf.extractall('/content/fresh')
for p in glob.glob('/content/fresh/**/*.*', recursive=True):
    if p.lower().endswith(('.jpg','.jpeg','.png')): INDEX[os.path.splitext(os.path.basename(p))[0]] = p
assert not (set(FR.image_id) - set(INDEX)) and not (set(splits['train'].image_id.astype(str)) - set(INDEX)), 'images missing'
SUB = np.sort(np.random.default_rng(7).choice(FR.image_id.to_numpy(), 2000, replace=False))
BASE224 = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])
def load_u8(i): return np.asarray(BASE224(Image.open(INDEX[i]).convert('RGB')), dtype=np.uint8)
CLEAN = np.stack([load_u8(i) for i in SUB])   # 2000 x 224 x 224 x 3 uint8
def jpeg(a, q):
    b = io.BytesIO(); Image.fromarray(a).save(b, 'JPEG', quality=q); return np.asarray(Image.open(b).convert('RGB'))
def blur(a, r): return np.asarray(Image.fromarray(a).filter(ImageFilter.GaussianBlur(r)))
yy, xx = np.mgrid[:224, :224]; VIG = (1 - 0.6 * (((yy - 112) ** 2 + (xx - 112) ** 2) / (2 * 112 ** 2))).clip(0.3, 1)[..., None]
PERTURB = {   # name: (group, function on uint8 HxWx3)
 'lowpass_r2':  ('frequency',   lambda a: blur(a, 2)),
 'lowpass_r4':  ('frequency',   lambda a: blur(a, 4)),
 'highpass_r4': ('frequency',   lambda a: np.clip(a.astype(np.int16) - blur(a, 4).astype(np.int16) + 128, 0, 255).astype(np.uint8)),
 'drop_red':    ('channel',     lambda a: np.concatenate([np.full_like(a[..., :1], int(a[..., 0].mean())), a[..., 1:]], -1)),
 'drop_green':  ('channel',     lambda a: np.concatenate([a[..., :1], np.full_like(a[..., 1:2], int(a[..., 1].mean())), a[..., 2:]], -1)),
 'drop_blue':   ('channel',     lambda a: np.concatenate([a[..., :2], np.full_like(a[..., 2:], int(a[..., 2].mean()))], -1)),
 'jpeg_q15':    ('degradation', lambda a: jpeg(a, 15)),
 'gamma_dark':  ('degradation', lambda a: (255 * (a / 255.0) ** 1.8).astype(np.uint8)),
 'gamma_bright':('degradation', lambda a: (255 * (a / 255.0) ** 0.55).astype(np.uint8)),
 'noise':       ('degradation', lambda a: np.clip(a + np.random.default_rng(0).normal(0, 20, a.shape), 0, 255).astype(np.uint8)),
 'vignette':    ('degradation', lambda a: (a * VIG).astype(np.uint8)),
 'blur_r3':     ('degradation', lambda a: blur(a, 3)),
}
PERT = {k: np.stack([f(a) for a in CLEAN]) for k, (_, f) in PERTURB.items()}
print(len(FR), 'confirmation-set eyes;', len(SUB), 'fingerprint images;', len(PERT), 'perturbations')

15794 confirmation-set eyes; 2000 fingerprint images; 12 perturbations


In [4]:
# Cell 4: training and prediction helpers
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
TF_TRAIN = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                               transforms.ColorJitter(0.2, 0.2, 0.1), transforms.ToTensor(), transforms.Normalize(MEAN.flatten().tolist(), STD.flatten().tolist())])
TF_EVAL = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(MEAN.flatten().tolist(), STD.flatten().tolist())])
class DS(Dataset):
    def __init__(self, ids, labels, tf): self.ids, self.labels, self.tf = list(ids), list(labels), tf
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return self.tf(Image.open(INDEX[str(self.ids[i])]).convert('RGB')), int(self.labels[i]), i
def score(out): return torch.softmax(out.float(), 1)[:, 2:].sum(1)
@torch.no_grad()
def predict_ids(m, ids, bs=128):
    m.eval(); p = np.zeros(len(ids))
    for x, _, idx in DataLoader(DS(ids, [0] * len(ids), TF_EVAL), batch_size=bs, num_workers=NW, pin_memory=True):
        with torch.amp.autocast('cuda'): p[idx.numpy()] = score(m(x.to(dev))).double().cpu().numpy()
    return p
def train_zoo_model(name):
    r = RECIPE[Z.loc[name, 'recipe']]; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    m = build(name).to(dev); tr, va = splits['train'], splits['val']
    g = torch.Generator(); g.manual_seed(SEED)
    dl_ = DataLoader(DS(tr.image_id.astype(str), tr.level, TF_TRAIN), batch_size=r['bs'], shuffle=True, drop_last=True, num_workers=NW, generator=g,
                     pin_memory=True, persistent_workers=True, prefetch_factor=4)
    opt = torch.optim.AdamW(m.parameters(), lr=r['lr'], weight_decay=r['wd']); E, W = r['epochs'], r['warmup']
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda ep: (ep + 1) / W if ep < W else 0.5 * (1 + math.cos(math.pi * (ep - W) / max(1, E - W))))
    scaler = torch.amp.GradScaler('cuda'); lossf = nn.CrossEntropyLoss(); best = (-1, None, 0)
    for ep in range(E):
        m.train(); t0 = time.time()
        for x, y, _ in dl_:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'): loss = lossf(m(x), y)
            scaler.scale(loss).backward()
            if r['clip']: scaler.unscale_(opt); nn.utils.clip_grad_norm_(m.parameters(), r['clip'])
            scaler.step(opt); scaler.update()
        sch.step(); vauc = roc_auc_score((va.level >= 2).astype(int), predict_ids(m, va.image_id.astype(str)))
        if vauc > best[0]: best = (vauc, {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}, ep + 1)
        print(f'   epoch {ep+1}/{E} val_auc {vauc:.4f} ({time.time()-t0:.0f}s)')
    m.load_state_dict(best[1]); return m, best[0], best[2]
print('ready')

ready


In [5]:
# Cell 5: STAGE 2, train the zoo and predict the confirmation set (resumable)
REC = f'{ZOO_OUT}/zoo_training_record.json'; rec = json.load(open(REC)) if os.path.exists(REC) else {}
for name in Z.index:
    ck, pf = f'{ZOO_CK}/{name}.pt', f'{ZOO_OUT}/preds/{name}_fresh.csv'
    if os.path.exists(ck) and os.path.exists(pf): continue
    print(f'=== {name} ({Z.loc[name, "timm_id"]}) ==='); t0 = time.time()
    m, vauc, ep = train_zoo_model(name)
    torch.save({'state_dict': m.state_dict(), 'name': name, 'val_auc': vauc, 'best_epoch': ep}, ck)
    p = predict_ids(m, FR.image_id)
    pd.DataFrame({'image_id': FR.image_id, 'predicted_probability': p}).to_csv(pf, index=False, float_format='%.17g')
    rec[name] = {'val_auc': round(float(vauc), 4), 'best_epoch': ep, 'fresh_auc': round(float(roc_auc_score(FR.binary_label, p)), 4), 'minutes': round((time.time() - t0) / 60, 1)}
    json.dump(rec, open(REC, 'w'), indent=2); print(f'   done {rec[name]}'); del m; torch.cuda.empty_cache()
print(pd.DataFrame(rec).T.sort_values('fresh_auc', ascending=False).to_string())

=== resnet50 (resnet50) ===
   epoch 1/5 val_auc 0.7331 (91s)
   epoch 2/5 val_auc 0.7643 (37s)
   epoch 3/5 val_auc 0.8063 (38s)
   epoch 4/5 val_auc 0.8210 (37s)
   epoch 5/5 val_auc 0.8196 (37s)
   done {'val_auc': 0.821, 'best_epoch': 4, 'fresh_auc': 0.824, 'minutes': 4.8}
=== resnet50_swsl (resnet50.fb_swsl_ig1b_ft_in1k) ===
   epoch 1/5 val_auc 0.8383 (38s)
   epoch 2/5 val_auc 0.8735 (37s)
   epoch 3/5 val_auc 0.8828 (38s)
   epoch 4/5 val_auc 0.8983 (37s)
   epoch 5/5 val_auc 0.9036 (37s)
   done {'val_auc': 0.9036, 'best_epoch': 5, 'fresh_auc': 0.9118, 'minutes': 3.6}
=== resnext50 (resnext50_32x4d.a1h_in1k) ===
   epoch 1/5 val_auc 0.7835 (77s)
   epoch 2/5 val_auc 0.8277 (37s)
   epoch 3/5 val_auc 0.8345 (37s)
   epoch 4/5 val_auc 0.8539 (37s)
   epoch 5/5 val_auc 0.8522 (37s)
   done {'val_auc': 0.8539, 'best_epoch': 4, 'fresh_auc': 0.8612, 'minutes': 4.5}
=== regnety_032 (regnety_032.ra_in1k) ===
   epoch 1/5 val_auc 0.8199 (120s)
   epoch 2/5 val_auc 0.8619 (41s)
   epoch

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

   epoch 1/5 val_auc 0.8395 (63s)
   epoch 2/5 val_auc 0.8627 (63s)
   epoch 3/5 val_auc 0.8661 (63s)
   epoch 4/5 val_auc 0.8787 (63s)
   epoch 5/5 val_auc 0.8799 (63s)
   done {'val_auc': 0.8799, 'best_epoch': 5, 'fresh_auc': 0.882, 'minutes': 5.8}
                 val_auc  best_epoch  fresh_auc  minutes
vgg16_bn          0.9268         5.0     0.9273      4.6
convnext_t        0.9108         5.0     0.9240      4.6
convnext_t22k     0.9169         4.0     0.9237      3.7
swin_t            0.8955         5.0     0.9143      3.9
resnet50_swsl     0.9036         5.0     0.9118      3.6
beitv2_b          0.8848         5.0     0.8890      3.8
vit_b             0.8794         5.0     0.8847      3.7
vit_s             0.8823         5.0     0.8840      3.7
retfound          0.8799         5.0     0.8820      5.8
regnety_032       0.8820         4.0     0.8805      5.6
densenet121       0.8704         4.0     0.8782      7.3
deit_s            0.8708         5.0     0.8757      3.7
mixer_b1

In [6]:
# Cell 6: STAGE 3, outcomes and the calibration gate
from scipy.stats import spearmanr, rankdata
G = FR.level.to_numpy(); n = len(G); referable, sev, pdr = G >= 2, G >= 3, G == 4
def refer(s, f=0.25):
    k = int(round(f * len(s))); r = np.zeros(len(s), bool); r[np.argsort(-s, kind='stable')[:k]] = True; return r
S = {nm: pd.read_csv(f'{ZOO_OUT}/preds/{nm}_fresh.csv', dtype={'image_id': str}).set_index('image_id').predicted_probability.reindex(FR.image_id).to_numpy() for nm in Z.index}
REF = {nm: refer(s) for nm, s in S.items()}; RK = {nm: rankdata(s) / n for nm, s in S.items()}
def comiss(ra, rb, tgt):
    a, b = tgt & ~ra, tgt & ~rb; T = tgt.sum()
    return ((a & b).sum() / T) / ((a.sum() / T) * (b.sum() / T)) if a.sum() and b.sum() else np.nan
NAMES = list(Z.index); PAIRS = list(itertools.combinations(NAMES, 2))
OUT = pd.DataFrame([{'a': a, 'b': b, 'comiss_sev': comiss(REF[a], REF[b], sev), 'comiss_pdr': comiss(REF[a], REF[b], pdr),
                     'comiss_ref': comiss(REF[a], REF[b], referable)} for a, b in PAIRS])
SEVMISS = {nm: int((sev & ~REF[nm]).sum()) for nm in NAMES}
MV = {}
for a in NAMES:
    for b in NAMES:
        if a != b:
            e = refer(np.maximum(RK[a], RK[b])); MV[(a, b)] = int((sev & ~REF[a]).sum() - (sev & ~e).sum())
full = {}
for nm, fullname in ANCHOR_FULL.items():
    df = pd.read_csv(f'{PREDS}/eyepacs_predictions_fresh_{fullname}_v2.csv', dtype={'image_id': str})
    pc = next(c for c in ('predicted_probability','probability','prob') if c in df.columns)
    full[nm] = refer(df.drop_duplicates('image_id').set_index('image_id')[pc].astype(float).reindex(FR.image_id).to_numpy())
AP = list(itertools.combinations(ANCHOR_FULL, 2))
cheap = [OUT[((OUT.a == a) & (OUT.b == b)) | ((OUT.a == b) & (OUT.b == a))].comiss_sev.iloc[0] for a, b in AP]
fullv = [comiss(full[a], full[b], sev) for a, b in AP]
GATE = {'n_anchor_pairs': len(AP), 'spearman_cheap_vs_full_severe_comiss': round(float(spearmanr(cheap, fullv)[0]), 3)}
GATE['passed'] = GATE['spearman_cheap_vs_full_severe_comiss'] >= 0.6
json.dump(GATE, open(f'{ZOO_OUT}/calibration_gate.json', 'w'), indent=2); print('CALIBRATION GATE:', GATE)
if not GATE['passed']: print('Gate failed: per the plan, the cheap zoo is not a valid proxy. Stop here and send calibration_gate.json.')

CALIBRATION GATE: {'n_anchor_pairs': 28, 'spearman_cheap_vs_full_severe_comiss': -0.081, 'passed': False}
Gate failed: per the plan, the cheap zoo is not a valid proxy. Stop here and send calibration_gate.json.


In [7]:
# Cell 7: STAGE 4, fingerprints (no labels). Trained: perturbation response vectors, occlusion maps, penultimate features. Pretrained: features and feature shifts.
def to_tensor(u8): return ((torch.from_numpy(u8).permute(0, 3, 1, 2).float() / 255.0) - MEAN) / STD
@torch.no_grad()
def run_batches(fn, arr, bs=100):
    out = []
    for s in range(0, len(arr), bs):
        with torch.amp.autocast('cuda'): out.append(fn(to_tensor(arr[s:s + bs]).to(dev)).float().cpu())
    return torch.cat(out).numpy().astype(np.float64)
OCC_N, GRID = 200, 8; P = 224 // GRID
@torch.no_grad()
def occlusion_maps(m):
    maps = np.zeros((OCC_N, GRID * GRID))
    for i in range(OCC_N):
        x = to_tensor(CLEAN[i:i + 1]).to(dev); base = score(m(x)).item()
        xs = x.repeat(GRID * GRID, 1, 1, 1)
        for k in range(GRID * GRID):
            r, c = divmod(k, GRID); xs[k, :, r * P:(r + 1) * P, c * P:(c + 1) * P] = 0
        with torch.amp.autocast('cuda'): maps[i] = base - score(m(xs)).float().cpu().numpy()
    return maps
def pretrained_feats(name, arr):
    m = build(name, pretrained=True, nc=0 if Z.loc[name, 'timm_id'] != 'retfound' else 5).to(dev).eval()
    if isinstance(m, RF): f = lambda x: m.base(pixel_values=x).last_hidden_state[:, 1:].mean(1)
    elif Z.loc[name, 'recipe'] == 'tfm' and hasattr(m, 'num_prefix_tokens') and not Z.loc[name, 'timm_id'].startswith(('swin', 'mixer')):
        npt = m.num_prefix_tokens; f = lambda x: m.forward_features(x)[:, npt:].mean(1)
    else: f = lambda x: m(x)
    out = {'clean': run_batches(f, arr)}
    for k in ('lowpass_r4', 'highpass_r4', 'drop_green', 'jpeg_q15', 'vignette'): out[k] = run_batches(f, PERT[k])
    del m; torch.cuda.empty_cache(); return out
for name in Z.index:
    fp = f'{ZOO_OUT}/fingerprints/{name}.npz'
    if os.path.exists(fp): continue
    t0 = time.time(); m = build(name, pretrained=False).to(dev); m.load_state_dict(torch.load(f'{ZOO_CK}/{name}.pt', map_location='cpu', weights_only=False)['state_dict']); m.eval()
    clean_s = run_batches(lambda x: score(m(x)), CLEAN)
    resp = {k: run_batches(lambda x: score(m(x)), PERT[k]) - clean_s for k in PERTURB}
    feats = run_batches(lambda x: penultimate(m, x), CLEAN); occ = occlusion_maps(m)
    del m; torch.cuda.empty_cache(); pt = pretrained_feats(name, CLEAN)
    pt_shift = {k: np.linalg.norm(pt[k] - pt['clean'], axis=1) / (np.linalg.norm(pt['clean'], axis=1) + 1e-8) for k in pt if k != 'clean'}
    np.savez_compressed(fp, clean_score=clean_s, feats=feats, occ=occ, pt_feats=pt['clean'],
                        **{f'resp_{k}': v for k, v in resp.items()}, **{f'ptshift_{k}': v for k, v in pt_shift.items()})
    print(f'{name:15s} fingerprints {time.time()-t0:.0f}s')

resnet50        fingerprints 69s
resnet50_swsl   fingerprints 27s
resnext50       fingerprints 61s
regnety_032     fingerprints 94s
densenet121     fingerprints 138s
efficientnet_b3 fingerprints 136s
effnetv2_s      fingerprints 101s
mobilenetv3     fingerprints 105s
convnext_t      fingerprints 57s
convnext_t22k   fingerprints 31s
inception_v3    fingerprints 112s
vgg16_bn        fingerprints 63s
vit_b           fingerprints 35s
vit_b_mae       fingerprints 34s
vit_b_dinov2    fingerprints 42s
vit_b_clip      fingerprints 35s
vit_s           fingerprints 24s
deit_s          fingerprints 22s
swin_t          fingerprints 38s
beitv2_b        fingerprints 38s
mixer_b16       fingerprints 35s


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

retfound        fingerprints 76s


In [8]:
# Cell 8: STAGE 5a, pair table of fingerprint similarities
FPX = {nm: dict(np.load(f'{ZOO_OUT}/fingerprints/{nm}.npz')) for nm in NAMES}
def cka(X, Y):
    X = X - X.mean(0); Y = Y - Y.mean(0); K, L = X @ X.T, Y @ Y.T
    return float((K * L).sum() / (np.linalg.norm(K) * np.linalg.norm(L)))
def rho(u, v): return float(spearmanr(u, v)[0])
GROUPS = {g: [k for k, (gg, _) in PERTURB.items() if gg == g] for g in ('frequency', 'channel', 'degradation')}
rows = []
for a, b in PAIRS:
    A, B = FPX[a], FPX[b]; r = {'a': a, 'b': b}
    for g, ks in GROUPS.items(): r[f'sim_{g}'] = np.mean([rho(A[f'resp_{k}'], B[f'resp_{k}']) for k in ks])
    r['sim_occlusion'] = np.nanmean([rho(A['occ'][i], B['occ'][i]) for i in range(OCC_N)])
    r['sim_clean_scores'] = rho(A['clean_score'], B['clean_score'])
    r['cka_trained'] = cka(A['feats'], B['feats']); r['cka_pretrained'] = cka(A['pt_feats'], B['pt_feats'])
    r['sim_pretrained_shift'] = np.mean([rho(A[k], B[k]) for k in A if k.startswith('ptshift_')])
    r['same_family'] = int(Z.loc[a, 'family'] == Z.loc[b, 'family']); r['same_data'] = int(Z.loc[a, 'pretrain_data'] == Z.loc[b, 'pretrain_data'])
    r['same_objective'] = int(Z.loc[a, 'objective'] == Z.loc[b, 'objective']); r['same_recipe'] = int(Z.loc[a, 'recipe'] == Z.loc[b, 'recipe'])
    r['mean_sevmiss'] = (SEVMISS[a] + SEVMISS[b]) / 2; r['gap_sevmiss'] = abs(SEVMISS[a] - SEVMISS[b])
    rows.append(r)
PT = pd.DataFrame(rows).merge(OUT, on=['a', 'b']); PT.to_csv(f'{ZOO_OUT}/pair_table.csv', index=False)
FPRED = ['sim_frequency','sim_channel','sim_degradation','sim_occlusion','cka_trained','cka_pretrained','sim_pretrained_shift','sim_clean_scores']
print(PT[FPRED + ['comiss_sev']].corr(method='spearman')['comiss_sev'].round(3).to_string())

sim_frequency           0.842
sim_channel             0.871
sim_degradation         0.863
sim_occlusion           0.611
cka_trained             0.865
cka_pretrained         -0.038
sim_pretrained_shift    0.005
sim_clean_scores        0.856
comiss_sev              1.000


In [9]:
# Cell 9: STAGE 5b, prediction with model-level intervals, leave-one-family-out, added value, and the clinical partner-choice test
from sklearn.linear_model import RidgeCV
Y = 'comiss_sev'; COV = ['mean_sevmiss', 'gap_sevmiss']; SPEC = ['same_family', 'same_data', 'same_objective', 'same_recipe']
LABEL_FREE = [c for c in FPRED if c != 'sim_clean_scores']      # pre-deployment fingerprints (clean-score similarity is test-firing, reported separately)
D = PT.dropna(subset=[Y]).copy(); D['logY'] = np.log(D[Y])
rng = np.random.default_rng(3); UNI = {}
for p in FPRED + SPEC:
    obs = rho(D[p], D[Y]); bs = []
    for _ in range(500):
        keep = set(rng.choice(NAMES, len(NAMES), replace=True)); sub = D[D.a.isin(keep) & D.b.isin(keep)]
        if len(sub) > 10: bs.append(rho(sub[p], sub[Y]))
    UNI[p] = {'rho': round(obs, 3), 'ci95': [round(float(np.nanpercentile(bs, 2.5)), 3), round(float(np.nanpercentile(bs, 97.5)), 3)]}
def lofo(features):
    pred = pd.Series(np.nan, index=D.index)
    for fam in Z.family.unique():
        held = set(Z.index[Z.family == fam]); te = D.a.isin(held) | D.b.isin(held); tr = ~te
        if te.sum() == 0 or tr.sum() < 20: continue
        X = D[features].values; mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-9
        mdl = RidgeCV(alphas=np.logspace(-3, 3, 13)).fit((X[tr] - mu) / sd, D.logY[tr]); pred[te] = mdl.predict((X[te] - mu) / sd)
    ok = pred.notna(); return pred, round(rho(pred[ok], D.logY[ok]), 3)
MODELS_ = {'accuracy_only': COV, 'spec_sheet': COV + SPEC, 'fingerprints': COV + LABEL_FREE, 'fingerprints_plus_spec': COV + SPEC + LABEL_FREE,
           'test_firing_only': COV + ['sim_clean_scores']}
LOFO = {k: lofo(v) for k, v in MODELS_.items()}
print('leave-one-family-out rank correlation (predicted vs observed severe co-miss):', {k: v[1] for k, v in LOFO.items()})
# Clinical test: for each deployed model, partner chosen by lowest predicted co-miss (from the fingerprint model with its family held out) among partners within 10% of the best severe-miss count, vs the strongest partner
predF = LOFO['fingerprints'][0]
def pred_pair(a, b):
    r = D[((D.a == a) & (D.b == b)) | ((D.a == b) & (D.b == a))]; return float(predF[r.index[0]]) if len(r) and not np.isnan(predF[r.index[0]]) else np.inf
CLIN = []
for a in NAMES:
    cands = [b for b in NAMES if b != a]; best = min(SEVMISS[b] for b in cands); tier = [b for b in cands if SEVMISS[b] <= 1.10 * best]
    strongest = min(cands, key=lambda b: SEVMISS[b]); fp_pick = min(tier, key=lambda b: pred_pair(a, b))
    CLIN.append({'anchor': a, 'strongest': strongest, 'fingerprint_pick': fp_pick, 'mv_strongest': MV[(a, strongest)], 'mv_fingerprint': MV[(a, fp_pick)],
                 'oracle': max(MV[(a, b)] for b in cands)})
CL = pd.DataFrame(CLIN); d = (CL.mv_fingerprint - CL.mv_strongest).to_numpy()
bs = [rng.choice(d, len(d), replace=True).mean() for _ in range(2000)]
CLIN_SUM = {'mean_extra_severe_rescued_vs_strongest': round(float(d.mean()), 2), 'ci95': [round(float(np.percentile(bs, 2.5)), 2), round(float(np.percentile(bs, 97.5)), 2)],
            'anchors_better': int((d > 0).sum()), 'anchors_worse': int((d < 0).sum()), 'mean_oracle_minus_strongest': round(float((CL.oracle - CL.mv_strongest).mean()), 2)}
print('clinical partner-choice test:', CLIN_SUM)

/tmp/ipykernel_4128/1655685954.py:6: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  def rho(u, v): return float(spearmanr(u, v)[0])


leave-one-family-out rank correlation (predicted vs observed severe co-miss): {'accuracy_only': 0.874, 'spec_sheet': 0.84, 'fingerprints': 0.911, 'fingerprints_plus_spec': 0.913, 'test_firing_only': 0.907}
clinical partner-choice test: {'mean_extra_severe_rescued_vs_strongest': 0.0, 'ci95': [0.0, 0.0], 'anchors_better': 0, 'anchors_worse': 0, 'mean_oracle_minus_strongest': 1.27}


In [10]:
# Cell 10: write results and figure
import matplotlib as mpl; mpl.use('Agg'); import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 4, figsize=(17, 8)); FAM_COL = {f: c for f, c in zip(sorted(Z.family.unique()), plt.cm.tab20.colors)}
for ax, p in zip(axes.ravel(), FPRED):
    same = D.same_family == 1
    ax.scatter(D[p][~same], D[Y][~same], s=16, c='#0072B2', alpha=0.55, label='different family'); ax.scatter(D[p][same], D[Y][same], s=26, c='#D55E00', label='same family')
    ax.set_yscale('log'); ax.set_xlabel(p.replace('_', ' ')); ax.set_title(f"rho {UNI[p]['rho']:+.2f} {UNI[p]['ci95']}", loc='left', fontsize=9, weight='bold')
    ax.spines[['top', 'right']].set_visible(False)
axes[0, 0].set_ylabel('Severe co-miss ratio'); axes[1, 0].set_ylabel('Severe co-miss ratio'); axes[0, 0].legend(fontsize=7, frameon=False)
fig.suptitle('Ballistics pilot: which fingerprints of how a model sees predict shared severe-disease misses?', x=0.01, ha='left', fontsize=12, weight='bold')
fig.tight_layout(); fig.savefig(f'{BASE}/05_figures/ballistics_pilot_v2.png', dpi=220, bbox_inches='tight'); plt.close(fig)
res = {'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'plan': 'ballistics_pilot_plan.json', 'status': 'exploratory',
       'zoo_training': json.load(open(REC)), 'calibration_gate': GATE, 'univariate_spearman_vs_severe_comiss': UNI,
       'leave_one_family_out_rank_correlation': {k: v[1] for k, v in LOFO.items()}, 'clinical_partner_choice': CLIN_SUM,
       'clinical_detail': CL.to_dict(orient='records'), 'n_models': len(NAMES), 'n_pairs': int(len(D))}
json.dump(res, open(f'{ZOO_OUT}/ballistics_pilot_results.json', 'w'), indent=2, default=float)
print('Wrote 04_results/ballistics/ballistics_pilot_results.json, pair_table.csv, and 05_figures/ballistics_pilot_v2.png')

Wrote 04_results/ballistics/ballistics_pilot_results.json, pair_table.csv, and 05_figures/ballistics_pilot_v2.png
